In [34]:
import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve()))

In [36]:
import importlib
import src.prompts
importlib.reload(src.prompts)

<module 'src.prompts' from 'C:\\Users\\digvi\\Videos\\pr-previewer\\src\\prompts.py'>

In [37]:
from src.data_utils import load_chunks
from src.loaders import load_vectorstore,load_reranker
from src.retrievers import create_bm25,create_hybrid_retriever
from src.prompts import format_context,RAG_PROMPT
from src.config import llm
from langchain_core.output_parsers import StrOutputParser
from src.config import RERANK_TOP_K

In [38]:
chunks=load_chunks()

vectorstore=load_vectorstore()

bm25=create_bm25(chunks)

hybrid_retriever=create_hybrid_retriever(bm25,vectorstore)

reranker=load_reranker()

llm=llm

Loading weights: 100%|█████████████████████████████████████████████████████████████| 105/105 [00:00<00:00, 5240.20it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [49]:
query="what is the name of teacher that is incharge of the bfsi mini project ?"

In [50]:
retrieved_docs=hybrid_retriever.invoke(query)

print(f"Retrievded  {len(retrieved_docs)} documents")

Retrievded  19 documents


In [51]:
# reranking

pairs=[[query,doc.page_content] for doc in retrieved_docs]

scores=reranker.predict(pairs)

In [52]:
# sorting in descending orders

ranked_docs=sorted(zip(retrieved_docs,scores),key=lambda x:x[1],reverse=True)

In [53]:
top_docs=[doc for doc,score in ranked_docs[:RERANK_TOP_K]]

In [54]:
# building the context in the form of one string 
context=format_context(top_docs)
print(context)


Chunk 1
Module 1 and 2 Assignment Report: Computer Networks 
Administrative Details 
● Department: Artificial Intelligence and Data Science (AI & DS) 
● Activity: Assignment for Module 1 and Module 2 
● Subject Faculty: B.S. Kukarni 
● Submission Date: 11-04-26 
● Student Name: Bhavesh Darbarsing Girase (24122) 
 
1. Network Types and AI-Based Applications 
 Computer networks are categorized by their geographical span and connectivity 
methods.

Chunk 2
LayerNorm(x + Sublayer(x)), where Sublayer(x) is the function implemented by the sub-layer
itself. To facilitate these residual connections, all sub-layers in the model, as well as the embedding
layers, produce outputs of dimension dmodel = 512.
Decoder: The decoder is also composed of a stack of N = 6 identical layers. In addition to the two
sub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head
attention over the output of the encoder stack. Similar to the encoder, we employ residual connec

In [55]:
parser=StrOutputParser()

In [56]:
# chain 
rag_chain= RAG_PROMPT | llm | parser

In [57]:
response=rag_chain.invoke({"context":context,"question":query})

In [58]:
response

"I don't have enough information to answer that question."